# Sesión 08 — Métodos de Árbol y Ensambles
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo II · Modelos Discriminativos**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Derivar los criterios de división de árboles de decisión (impureza de Gini, entropía) y visualizar el sobreajuste en función de la profundidad.
2. Explicar cómo el bagging y la aleatoriedad de características del Random Forest reducen la varianza y calcular el error OOB.
3. Implementar el algoritmo de Gradient Boosting por etapas y entender cómo XGBoost lo mejora.
4. Calcular e interpretar valores SHAP globales (summary plot) y locales (waterfall plot) para explicar predicciones individuales de mortalidad en UCI.
5. Comparar importancias MDI, por permutación y SHAP, y entender cuándo discrepan.

## Conjunto de datos principal

**PhysioNet Challenge 2012 — Mortalidad en UCI**
Silva, I. et al. (2012). Predicting in-hospital mortality of ICU patients: the PhysioNet/Computing in Cardiology Challenge 2012.
*Computing in Cardiology*, 39, 245–248. https://physionet.org/content/challenge-2012/

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning* (2ª ed.). §9.2 (Árboles), §15 (Random Forests), §10 (Boosting). Springer. |
| ★★★ | Lundberg, S.M. & Lee, S.I. (2017). A unified approach to interpreting model predictions. *NeurIPS 2017*. https://arxiv.org/abs/1705.07874 |
| ★★☆ | Chen, T. & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *KDD 2016*. https://doi.org/10.1145/2939672.2939785 |
| ★★☆ | Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324 |
| ★☆☆ | Friedman, J.H. (2001). Greedy function approximation: a gradient boosting machine. *Annals of Statistics*, 29(5), 1189–1232. |

## Parte 0 — Configuración y datos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})

# ── Dataset UCI CinC 2012 (mismo generador que S06 y S07) ─────────────────────
# Silva, I. et al. (2012). Computing in Cardiology, 39, 245–248.
# https://physionet.org/content/challenge-2012/
N = 1000
nombres_feats = ['Edad','APACHE-II','FC media','SpO2 media',
                 'Creatinina','Bilirrubina','Glasgow','FiO2']

def generar_uci(N, rng_):
    edad   = rng_.normal(63, 16, N).clip(18, 95)
    apache = rng_.normal(18, 8,  N).clip(0,  71)
    fc     = rng_.normal(88, 20, N).clip(40, 180)
    spo2   = rng_.normal(94, 5,  N).clip(60, 100)
    creat  = rng_.gamma(2, 0.8,  N).clip(0.3, 15)
    bili   = rng_.gamma(1.5, 0.6, N).clip(0.1, 20)
    glas   = rng_.normal(11, 4,  N).clip(3,  15)
    fio2   = rng_.beta(2, 5, N) * 0.6 + 0.21
    X = np.column_stack([edad, apache, fc, spo2, creat, bili, glas, fio2])
    logit = (-4.5 + 0.03*edad + 0.12*apache + 0.008*fc - 0.05*spo2
             + 0.15*creat + 0.08*bili - 0.10*glas + 2.0*fio2)
    p = 1 / (1 + np.exp(-logit))
    y = rng_.binomial(1, p).astype(float)
    return X.astype(np.float32), y

X_uci, y_uci = generar_uci(N, rng)

# División estratificada 75/25
idx_pos = np.where(y_uci == 1)[0]
idx_neg = np.where(y_uci == 0)[0]
n_tr_p  = int(len(idx_pos) * 0.75)
n_tr_n  = int(len(idx_neg) * 0.75)
idx_tr  = np.concatenate([rng.permutation(idx_pos)[:n_tr_p],
                            rng.permutation(idx_neg)[:n_tr_n]])
idx_te  = np.setdiff1d(np.arange(N), idx_tr)
X_tr, y_tr = X_uci[idx_tr], y_uci[idx_tr]
X_te, y_te = X_uci[idx_te], y_uci[idx_te]
print(f'UCI N={N} | prevalencia={y_uci.mean():.1%} | train={len(X_tr)} test={len(X_te)}')

## Parte 1 — Árboles de decisión: criterios de división y sobreajuste

Un árbol de decisión divide el espacio de características de forma recursiva.
En cada nodo se elige el corte $(j, t)$ que maximiza la reducción de impureza:

$$\Delta I = I(\text{nodo}) - \frac{n_L}{n}I(L) - \frac{n_R}{n}I(R)$$

| Criterio | Fórmula | Uso |
|---|---|---|
| **Gini** | $1 - \sum_k p_k^2$ | Clasificación (por defecto) |
| **Entropía** | $-\sum_k p_k \log_2 p_k$ | Clasificación (más costosa) |
| **MSE** | $\text{Var}(y)$ | Regresión |

In [ ]:
# ── Sobreajuste vs profundidad del árbol ──────────────────────────────────────
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
profundidades = range(1, 16)

auroc_tr, auroc_cv = [], []
for d in profundidades:
    arbol = DecisionTreeClassifier(max_depth=d, random_state=42,
                                    class_weight='balanced')
    arbol.fit(X_tr, y_tr)
    auroc_tr.append(roc_auc_score(y_tr, arbol.predict_proba(X_tr)[:, 1]))
    auroc_cv.append(cross_val_score(arbol, X_uci, y_uci,
                                     cv=cv5, scoring='roc_auc').mean())

idx_opt = np.argmax(auroc_cv)
d_opt   = list(profundidades)[idx_opt]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(profundidades, auroc_tr, 'steelblue', lw=2.5, label='Train')
axes[0].plot(profundidades, auroc_cv, 'tomato',    lw=2.5, label='CV (5-fold)')
axes[0].axvline(d_opt, color='gray', ls='--', lw=1.5, label=f'Profundidad óptima={d_opt}')
axes[0].set(xlabel='Profundidad máxima', ylabel='AUROC',
            title='Sobreajuste del árbol de decisión\nAUROC train vs validación cruzada')
axes[0].legend(fontsize=9)

# Visualizar el árbol óptimo (solo 3 niveles para legibilidad)
arbol_opt = DecisionTreeClassifier(max_depth=min(d_opt, 3), random_state=42,
                                    class_weight='balanced')
arbol_opt.fit(X_tr, y_tr)
plot_tree(arbol_opt, feature_names=nombres_feats,
          class_names=['Sobrevive','Muere'], filled=True,
          fontsize=8, ax=axes[1], impurity=True)
axes[1].set_title(f'Árbol de decisión (profundidad={min(d_opt, 3)})\n'
                   'UCI mortalidad — primeros 3 niveles', fontsize=10)
plt.tight_layout()
plt.show()

print(f'Profundidad óptima por CV: {d_opt}')
print(f'AUROC CV óptimo: {auroc_cv[idx_opt]:.3f}')

## Parte 2 — Random Forest: bagging y aleatoriedad de características

El Random Forest combina dos fuentes de aleatoriedad:

1. **Bagging:** cada árbol se entrena en una muestra bootstrap del conjunto de entrenamiento.
2. **Aleatoriedad de características:** en cada split, solo se considera un subconjunto aleatorio de `max_features` características.

Las muestras no incluidas en cada bootstrap (**out-of-bag, OOB**) sirven como conjunto de validación interno — el error OOB es un estimador prácticamente libre de sesgo del error de generalización.

In [ ]:
# ── Convergencia OOB y número de árboles ──────────────────────────────────────
n_estimadores = [1, 5, 10, 20, 50, 100, 200, 300]
oob_scores, te_scores = [], []

for n_est in n_estimadores:
    rf = RandomForestClassifier(n_estimators=n_est, max_features='sqrt',
                                 oob_score=True, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    oob_scores.append(rf.oob_score_)   # exactitud OOB (no AUROC)
    te_scores.append(roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1]))

# Modelo final con 200 árboles para importancias
rf_final = RandomForestClassifier(n_estimators=200, max_features='sqrt',
                                    oob_score=True, class_weight='balanced',
                                    random_state=42, n_jobs=-1)
rf_final.fit(X_tr, y_tr)

# Importancias MDI (Mean Decrease Impurity)
imp_mdi = rf_final.feature_importances_
idx_mdi = np.argsort(imp_mdi)[::-1]

# Importancias por permutación
perm = permutation_importance(rf_final, X_te, y_te,
                               n_repeats=20, random_state=42, n_jobs=-1,
                               scoring='roc_auc')
imp_perm = perm.importances_mean

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Convergencia
axes[0].plot(n_estimadores, te_scores, 'steelblue', lw=2.5, marker='o', ms=5,
              label='AUROC test')
axes[0].plot(n_estimadores, oob_scores, 'tomato', lw=2, ls='--', marker='s', ms=4,
              label='Exactitud OOB')
axes[0].set(xlabel='Número de árboles', ylabel='Puntuación',
            title='Convergencia del Random Forest\nOOB vs test')
axes[0].legend(fontsize=9)

# MDI
axes[1].barh(range(8), imp_mdi[idx_mdi][::-1], color='steelblue', alpha=0.8)
axes[1].set_yticks(range(8))
axes[1].set_yticklabels([nombres_feats[i] for i in idx_mdi[::-1]], fontsize=9)
axes[1].set(xlabel='Importancia MDI', title='Importancias de características\nMDI (Mean Decrease Impurity)')

# Permutación
idx_perm = np.argsort(imp_perm)[::-1]
axes[2].barh(range(8), imp_perm[idx_perm][::-1], color='seagreen', alpha=0.8,
              xerr=perm.importances_std[idx_perm][::-1], capsize=3)
axes[2].set_yticks(range(8))
axes[2].set_yticklabels([nombres_feats[i] for i in idx_perm[::-1]], fontsize=9)
axes[2].set(xlabel='Δ AUROC (permutación)', title='Importancias por permutación\n(más confiables que MDI)')

plt.suptitle('Random Forest — UCI CinC 2012', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

auc_rf = roc_auc_score(y_te, rf_final.predict_proba(X_te)[:, 1])
print(f'Random Forest (200 árboles) AUROC test: {auc_rf:.3f}')
print(f'Error OOB (exactitud): {1 - rf_final.oob_score_:.3f}')

## Parte 3 — Gradient Boosting y XGBoost

El Gradient Boosting construye modelos aditivos de forma voraz:

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu \cdot h_m(\mathbf{x})$$

donde $h_m$ es un árbol pequeño ajustado a los **residuos negativos del gradiente** de la pérdida.

**XGBoost** mejora esto añadiendo:
- Regularización L1/L2 sobre los pesos de las hojas
- Segunda derivada (Hessiano) en la expansión de Taylor de la pérdida
- Paralelismo y poda de árboles por ganancia

> **Referencia:** Chen, T. & Guestrin, C. (2016). XGBoost: A scalable tree boosting system.
> *KDD 2016*. https://doi.org/10.1145/2939672.2939785

In [ ]:
# ── Gradient Boosting por etapas y comparación con RF ─────────────────────────
gb = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=3,
    subsample=0.8, random_state=42
)
gb.fit(X_tr, y_tr)

# AUROC por etapa (staged_predict_proba)
auroc_staged = [
    roc_auc_score(y_te, p[:, 1])
    for p in gb.staged_predict_proba(X_te)
]

# XGBoost simulado con GBM bien ajustado (sin xgboost instalado)
# Usamos GradientBoosting con hiperparámetros de estilo XGBoost
xgb_sim = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, min_samples_leaf=5, random_state=42
)
xgb_sim.fit(X_tr, y_tr)
auroc_xgb = [
    roc_auc_score(y_te, p[:, 1])
    for p in xgb_sim.staged_predict_proba(X_te)
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# AUROC por ronda
axes[0].plot(auroc_staged, 'steelblue', lw=2, label='GradientBoosting')
axes[0].plot(auroc_xgb,    'tomato',    lw=2, label='GBM estilo XGBoost')
axes[0].axhline(auc_rf, color='seagreen', ls='--', lw=1.5,
                 label=f'Random Forest ({auc_rf:.3f})')
axes[0].set(xlabel='Número de árboles (rondas)', ylabel='AUROC (test)',
            title='AUROC por ronda de boosting\nUCI CinC 2012')
axes[0].legend(fontsize=9)

# Comparación de modelos hasta ahora
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

modelos = [
    ('Reg. Logística', Pipeline([('sc', StandardScaler()),
                                  ('clf', LogisticRegression(C=1, max_iter=500,
                                           class_weight='balanced'))])),
    ('SVM-RBF',        Pipeline([('sc', StandardScaler()),
                                  ('clf', SVC(kernel='rbf', C=1, gamma='scale',
                                           class_weight='balanced', probability=True))])),
    ('Árbol (óptimo)', DecisionTreeClassifier(max_depth=d_opt, random_state=42,
                                               class_weight='balanced')),
    ('Random Forest',  rf_final),
    ('GradBoost',      gb),
]

nombres_m, aurocs_m = [], []
for nombre, m in modelos:
    if nombre not in ('Random Forest', 'GradBoost', 'Árbol (óptimo)'):
        m.fit(X_tr, y_tr)
    elif nombre == 'Árbol (óptimo)':
        m.fit(X_tr, y_tr)
    a = roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])
    nombres_m.append(nombre); aurocs_m.append(a)
    print(f'{nombre:<20}  AUROC={a:.3f}')

colores = ['#3B82F6','#F59E0B','#EF4444','#10B981','#8B5CF6']
idx_s   = np.argsort(aurocs_m)
axes[1].barh([nombres_m[i] for i in idx_s],
              [aurocs_m[i] for i in idx_s],
              color=[colores[i] for i in idx_s], alpha=0.8)
axes[1].axvline(0.5, color='gray', ls='--', lw=1)
axes[1].set(xlabel='AUROC (test)', title='Comparación de modelos — Módulo II\n(hasta sesión 08)')
plt.tight_layout()
plt.show()

## Parte 4 — SHAP: explicabilidad para modelos de ensamble

Los valores SHAP (SHapley Additive exPlanations) atribuyen a cada característica su **contribución marginal promedio** a la predicción de cada instancia:

$$f(\mathbf{x}) = \mathbb{E}[f] + \sum_{j=1}^{d} \phi_j(\mathbf{x})$$

donde $\phi_j$ es el valor de Shapley de la característica $j$.

> **Referencia:** Lundberg, S.M. & Lee, S.I. (2017). A unified approach to interpreting model predictions.
> *NeurIPS 2017*. https://arxiv.org/abs/1705.07874

In [ ]:
# ── SHAP implementado desde cero (TreeSHAP aproximado) ────────────────────────
# Para un Random Forest, los valores SHAP se aproximan eficientemente mediante
# el algoritmo de intervención de árbol. Aquí implementamos una versión
# simplificada basada en permutaciones para mantener la independencia de librerías.

def shap_kernel_approx(model, X_background, X_explain, n_perm=100, rng_=None):
    """
    Aproximación de valores SHAP por KernelSHAP simplificado.
    Para cada instancia en X_explain calcula phi_j para cada característica j.
    """
    rng_ = rng_ or np.random.default_rng()
    n_explain, d = X_explain.shape
    E_f = model.predict_proba(X_background)[:, 1].mean()  # E[f(x)]
    phi = np.zeros((n_explain, d))

    for i in range(n_explain):
        x_i = X_explain[i]
        phi_i = np.zeros(d)
        for _ in range(n_perm):
            perm = rng_.permutation(d)
            # Tomar una instancia de referencia aleatoria del background
            x_ref = X_background[rng_.integers(len(X_background))].copy()
            x_prev = x_ref.copy()
            for j in perm:
                x_curr = x_prev.copy()
                x_curr[j] = x_i[j]
                # Marginal contribution of feature j
                phi_i[j] += (model.predict_proba(x_curr.reshape(1, -1))[0, 1] -
                              model.predict_proba(x_prev.reshape(1, -1))[0, 1])
                x_prev = x_curr
        phi[i] = phi_i / n_perm
    return phi, E_f

# Calcular SHAP para un subconjunto del test para eficiencia
n_shap    = 80
idx_shap  = rng.choice(len(X_te), n_shap, replace=False)
X_shap    = X_te[idx_shap]

print('Calculando valores SHAP (puede tomar ~30 s)...')
phi_vals, e_f = shap_kernel_approx(rf_final, X_tr[:100], X_shap,
                                    n_perm=50, rng_=rng)

# ── Summary plot (beeswarm) ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Global importance: |phi| medio
imp_shap   = np.abs(phi_vals).mean(axis=0)
idx_shap_s = np.argsort(imp_shap)[::-1]

# Summary beeswarm (scatter por característica)
y_jitter   = np.zeros((n_shap, 8))
for j in range(8):
    y_jitter[:, j] = idx_shap_s[j] + rng.uniform(-0.3, 0.3, n_shap)

# Normalizar valores de características para color
X_norm = (X_shap - X_shap.min(0)) / (X_shap.max(0) - X_shap.min(0) + 1e-8)

for rank, j in enumerate(idx_shap_s):
    sc = axes[0].scatter(
        phi_vals[:, j],
        np.full(n_shap, rank) + rng.uniform(-0.25, 0.25, n_shap),
        c=X_norm[:, j], cmap='RdBu_r', s=15, alpha=0.7, vmin=0, vmax=1
    )

axes[0].axvline(0, color='gray', lw=1)
axes[0].set_yticks(range(8))
axes[0].set_yticklabels([nombres_feats[j] for j in idx_shap_s], fontsize=9)
axes[0].set(xlabel='Valor SHAP (impacto en predicción)',
            title='SHAP Summary Plot\nRojo=valor alto, Azul=valor bajo')
plt.colorbar(sc, ax=axes[0], label='Valor de la característica (normalizado)')

# Waterfall para el paciente con mayor probabilidad predicha
pred_probs = rf_final.predict_proba(X_shap)[:, 1]
idx_worst  = np.argmax(pred_probs)
phi_worst  = phi_vals[idx_worst]
idx_sort   = np.argsort(np.abs(phi_worst))[::-1]

vals_sorted  = phi_worst[idx_sort]
feats_sorted = [nombres_feats[j] for j in idx_sort]
colores_w    = ['tomato' if v > 0 else 'steelblue' for v in vals_sorted]
cumsum       = np.concatenate([[e_f], e_f + np.cumsum(vals_sorted)])

axes[1].barh(range(8), vals_sorted, color=colores_w, alpha=0.85, left=cumsum[:8])
axes[1].axvline(e_f,               color='gray',  lw=1, ls='--', label=f'E[f]={e_f:.3f}')
axes[1].axvline(pred_probs[idx_worst], color='navy', lw=2, label=f'f(x)={pred_probs[idx_worst]:.3f}')
axes[1].set_yticks(range(8))
axes[1].set_yticklabels(feats_sorted, fontsize=9)
axes[1].set(xlabel='Probabilidad predicha de mortalidad',
            title=f'SHAP Waterfall — paciente de mayor riesgo\n'
                   f'P(muerte)={pred_probs[idx_worst]:.3f}')
axes[1].legend(fontsize=9)

plt.suptitle('Explicabilidad SHAP — Random Forest UCI CinC 2012', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print('\nTop 3 características por importancia SHAP global:')
for rank, j in enumerate(idx_shap_s[:3]):
    print(f'  {rank+1}. {nombres_feats[j]}: |φ| medio = {imp_shap[j]:.4f}')

## ✏️ Ejercicios

Los ejercicios usan el dataset **PTB-XL** — clasificación de ECG de 12 derivaciones.

> **Fuente:** Wagner, P. et al. (2020). PTB-XL, a large publicly available
> electrocardiography dataset. *Scientific Data*, 7, 154.
> https://doi.org/10.1038/s41597-020-0495-6
> https://physionet.org/content/ptb-xl/

```python
# Dataset PTB-XL simulado — características de 12 derivaciones
# Clases: Normal (0), Hipertrofia VI (1), Isquemia (2)
n_ptb, n_feats_ptb = 600, 24  # 2 características por derivación
y_ptb = rng.choice([0, 1, 2], n_ptb, p=[0.50, 0.25, 0.25])
medias_ptb = rng.normal(0, 1, (3, n_feats_ptb))
X_ptb = np.vstack([
    rng.normal(medias_ptb[k], 1.5, (np.sum(y_ptb==k), n_feats_ptb))
    for k in range(3)
]).astype(np.float32)
```

1. **Profundidad óptima por validación cruzada.** Grafica la curva AUROC one-vs-rest vs profundidad (1–15) para el dataset PTB-XL. ¿A qué profundidad comienza el sobreajuste? ¿Es la misma para las 3 clases?

2. **Efecto de `max_features`.** Entrena un Random Forest con `max_features` ∈ {1, 2, 4, 8, 12, 'sqrt', 'log2', None}. Grafica el AUROC OOB vs `max_features`. ¿Cuál es el valor óptimo para este dataset?

3. **Comparación de importancias.** Para el Random Forest óptimo del ejercicio 2, calcula las importancias MDI y por permutación. ¿Las dos ordenaciones coinciden? Identifica si hay características con importancia MDI alta pero permutación baja — ¿qué indica esto?

4. **Early stopping en GradBoost.** Entrena un GradientBoostingClassifier con 500 rondas y `validation_fraction=0.2`. Usa el AUROC de validación para determinar el número óptimo de árboles. ¿Cuántas rondas se necesitan? ¿Mejora vs Random Forest?

5. *(Desafío)* **SHAP en PTB-XL real.** Descarga el dataset PTB-XL (https://physionet.org/content/ptb-xl/). Extrae características de morfología del ECG (amplitud R, duración QRS, eje eléctrico por derivación). Entrena un Random Forest y calcula valores SHAP. ¿Qué derivaciones son más informativas para distinguir hipertrofia VI de isquemia?

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| PhysioNet CinC 2012 (UCI) | Silva, I. et al. (2012). *Computing in Cardiology*, 39, 245–248. https://physionet.org/content/challenge-2012/ | Dataset principal del Módulo II |
| PTB-XL ECG | Wagner, P. et al. (2020). *Scientific Data*, 7, 154. https://doi.org/10.1038/s41597-020-0495-6 | Ejercicios Sesión 08 |